# Pangenome Analysis Notebook

This notebook walks through a GPU accelerated pangenome analysis workflow using NVIDIA Parabricks. We start by downloading a pangenome graph and using filters to extract the reference path. Next we use the Parabricks Giraffe aligner to align FASTQ samples to this reference. Finally, we run the aligned samples through Parabricks DeepVariant for variant calling. 

### What is a pangenome? 

A pangenome is a comprehensive collection of all genomic sequences and variations present across multiple individuals or populations of a species, rather than relying on a single reference genome. Instead of treating one genome as the standard reference, pangenomes capture the genetic diversity by including sequences from many individuals, making it easier to identify variants and structural differences that might be missed in traditional single-reference approaches. Using pangenomes leads to better alignment and more accurate variant calling. 


## Step 1: Download the data

For this workflow, we start by downloading the FASTQ samples and the pangenome reference. 

Note: This will take about 30 mins to download. 

In [21]:
%%sh 

../scripts/download_pangenome.sh 

File ‘parabricks_sample.tar.gz’ already there; not retrieving.

File ‘hprc-v1.1-mc-grch38.dist’ already there; not retrieving.

File ‘hprc-v1.1-mc-grch38.min’ already there; not retrieving.

File ‘hprc-v1.1-mc-grch38.gbz’ already there; not retrieving.

parabricks_sample/
parabricks_sample/Data/
parabricks_sample/Data/markdup_input.bam
parabricks_sample/Data/sample_2.fq.gz
parabricks_sample/Data/sample_1.fq.gz
parabricks_sample/Data/single_ended.bam
parabricks_sample/Ref/
parabricks_sample/Ref/Homo_sapiens_assembly38.fasta.sa
parabricks_sample/Ref/Homo_sapiens_assembly38.known_indels.vcf.gz.tbi
parabricks_sample/Ref/Homo_sapiens_assembly38.dict
parabricks_sample/Ref/Homo_sapiens_assembly38.fasta
parabricks_sample/Ref/Homo_sapiens_assembly38.fasta.fai
parabricks_sample/Ref/Homo_sapiens_assembly38.fasta.pac
parabricks_sample/Ref/Homo_sapiens_assembly38.known_indels.vcf.gz
parabricks_sample/Ref/Homo_sapiens_assembly38.fasta.bwt
parabricks_sample/Ref/Homo_sapiens_assembly38.fasta.ann
parab

### FASTQ samples

These FASTQ samples contain short-read human genome data downsampled for testing purposes. 

Note: This code cell takes about 10 minutes to run. 

### Pangenome graph 

Pangenome data can stored in a few different ways, as outlined in the figure below. For our purposes we will use a pangenome graph. 

Source: [A gentle introduction to pangenomics](https://pmc.ncbi.nlm.nih.gov/articles/PMC11570541/#f2) 

![fq2bam_diagram](../images/pangenome_storage.png)

Specifically, we use the [HPRC](https://humanpangenome.org/) (Human Pangenome Reference Consortium) version 1.1 graph, which was constructed using [Minigraph-Cactus](https://github.com/ComparativeGenomicsToolkit/cactus/blob/master/doc/pangenome.md). We need the following files: 
- `.gbz` - The compressed graph 
- `.dist` - The distance index used for computing distances between points in the graph 
- `.min` - The minimizer index used to find minimizer substrings in the graph 

In [4]:
%%sh 

# Make the reference data directory if it doesn't exist
mkdir -p ../data/ref

# Download the data 
cd ../data/ref && \
    printf "%s\n" dist min gbz | xargs -I {} \
    wget -nc --progress=bar:force 2>&1 "https://s3-us-west-2.amazonaws.com/human-pangenomics/pangenomes/freeze/freeze1/minigraph-cactus/hprc-v1.1-mc-grch38/hprc-v1.1-mc-grch38.{}"

--2026-03-02 12:21:47--  https://s3-us-west-2.amazonaws.com/human-pangenomics/pangenomes/freeze/freeze1/minigraph-cactus/hprc-v1.1-mc-grch38/hprc-v1.1-mc-grch38.dist
Resolving s3-us-west-2.amazonaws.com (s3-us-west-2.amazonaws.com)... 52.92.162.80, 52.92.204.240, 52.92.197.0, ...
Connecting to s3-us-west-2.amazonaws.com (s3-us-west-2.amazonaws.com)|52.92.162.80|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 8694520760 (8.1G) [application/vnd.apple.installer+xml]
Saving to: ‘hprc-v1.1-mc-grch38.dist’

hprc-v1.1-mc-grch38 100%[===================>]   8.10G  42.8MB/s    in 3m 34s  

2026-03-02 12:25:21 (38.8 MB/s) - ‘hprc-v1.1-mc-grch38.dist’ saved [8694520760/8694520760]

--2026-03-02 12:25:21--  https://s3-us-west-2.amazonaws.com/human-pangenomics/pangenomes/freeze/freeze1/minigraph-cactus/hprc-v1.1-mc-grch38/hprc-v1.1-mc-grch38.min
Resolving s3-us-west-2.amazonaws.com (s3-us-west-2.amazonaws.com)... 52.218.153.128, 52.92.194.80, 52.92.185.168, ...
Connecting 

Now we must identify the reference path, which is the path through the graph that matches the human standard reference GRCh38 ([Genome Reference Consortium Human Build 38](https://genome.ucsc.edu/cgi-bin/hgGateway?token=0.eDjpSlexMn1Xh8rGqvYUpctc_8ZQwgdWcTDP0mg0e5YHk6b1gPw7TiSXXxKFNDG0F4gn8voyGzzIBzNT6aHsO2cylGlCg2UgoFVsH2vv9Qt2PBk7j3yHE-Bqey5qPNprFVirMzNU7meyTNF2r4qu6kwcgDgZEXC6lw_kgMuQfqkYxmhkgVOGgKfq7V3e7ixhPigalMgvKBCznLEf-qm3pWI74qDj-hOWHGBCykTmP9vd4Mj9fK8AAzLkY6LOVyr9lewYtUW9A-CRaa_BIUpoPxTOcj7QEkaLdmCn1pJytLmrkQXW64XSadhQzyI7REDt4M2-Lj3f6GkMMswIEG1ZOHJfBaK8UFPV1VaKzYduFCMr0z42RH4eg9qGQjdSPJxfMfK3bsydrMLcEba114kZWUJUG4GErA5ZW-y5kvjbwuh0osLhY9laHhdsPOBa61g8gRE_el82HxLFcX0-AuKOjjP7MGZJtJr08bu8_LxyWROLtMYOKW8IAA9sb79_0rHChM-TUxs96jAWXKjuVAzm_USOT1MFNSeDGeBmJ4kIrJkyoJB3wdQmLA4UaH83QW2q_VFXlgtwJjwY-ZY48HycsjVpZD_hhUWk_pJDuECJyazd5rOLVoWRCQzy4202zZsV5gUO9kaJqqsKe6dQfRz1RyIXoj_cxwcRs2ZZftF5N_KIXRQ_UduCJdAavB1kd_lh_QiBYieL-uUzLgMRwf_n1HCqZPgvduixF_N__-NOUEuz1j_7hZU3aTCv5x-i7oQMtluO2mVkSE8Vj8zIkpxB0404yKI4hLzKNfDj3KNjvCdR-XbksnaYjiuncgNwXiSroiScC_fwMubmv7Vj3Rm9DIEU7Ld3tXVkgh_SYYs2aTB34z3teD9ewBFJScb2Rgj_RfEnSWbC-IZjVZui5_B6AefNkQShaR4tIQM76iYA8Z8.NV4fHkVJjzg_jyWUdf5ORQ.5c8945b5c8156b8b333c2f2e4ac1371136e953526116bb6062a3d89aadefef88)). This helps to map our graph into a standard coordinate system that is used by other downstream tools and maintains compatability with tools that don't fully support graphs. 

For this we use the [`vg`](https://github.com/vgteam/vg) software package which has a tool called `paths`. The code cell below will find all the paths that start with `GRCh38` and save them to a file. 

In [5]:
%%sh 

docker run --rm \
    --volume $(pwd)/../data/ref:/workdir \
    --workdir /workdir \
    quay.io/vgteam/vg:v1.59.0 \
    vg paths \
        -x hprc-v1.1-mc-grch38.gbz \
        -L \
        -Q GRCh38 > ../data/ref/hprc-v1.1-mc-grch38.paths

Next we use `grep` to further filter this list to remove any unwanted paths. In this case, we filter out anything that's not Chromosomes 1-22 or Chromosomes X and Y. 

In [6]:
%%sh 

grep -v _decoy ../data/ref/hprc-v1.1-mc-grch38.paths \
    | grep -v _random \
    | grep -v chrUn_ \
    | grep -v chrEBV \
    | grep -v chrM \
    | grep -v chain_ > ../data/ref/hprc-v1.1-mc-grch38.paths.sub

Now we have the data downloaded and pre-processed, the data directory should have these files:  

```
data
├── ref
│   ├── hprc-v1.1-mc-grch38.dist
│   ├── hprc-v1.1-mc-grch38.gbz
│   ├── hprc-v1.1-mc-grch38.min
│   ├── hprc-v1.1-mc-grch38.paths
│   └── hprc-v1.1-mc-grch38.paths.sub
├── test_1.fastq.gz
└── test_2.fastq.gz

2 directories, 7 files
```

## Giraffe

Now that the samples are downloaded and the graph has been pre-processed, we are ready for short-read alignment using Giraffe. 

Giraffe was originally developed by Dr. Benedict Paten's lab at the University of California, Santa Cruz (UCSC). It brings many improvements over previous mapping methods, including haplotype-aware mapping, pre-calculate indexes, and now hardware acceleration on the GPU. Combined these allow for faster and more accurate results. For more details on Giraffe, see the [original paper](https://www.science.org/doi/10.1126/science.abg8871). 

In the next code cell, we will run the GPU-accelerated [Parabricks Giraffe](https://docs.nvidia.com/clara/parabricks/latest/documentation/tooldocs/man_giraffe.html). 

In [7]:
%%sh 

# Make the output directory if it doesn't exist 
mkdir -p ../data/out

# Run GPU-accelerated Giraffe using Parabricks 
docker run --rm --gpus all \
    --volume $(pwd)/../data:/workdir \
    --workdir /workdir \
    nvcr.io/nvidia/clara/clara-parabricks:4.6.0-1 \
    pbrun giraffe \
        --read-group "sample_rg1" \
        --sample "sample-name" \
        --read-group-library "library" \
        --read-group-platform "platform" \
        --read-group-pu "pu" \
        --dist-name ref/hprc-v1.1-mc-grch38.dist \
        --minimizer-name ref/hprc-v1.1-mc-grch38.min \
        --gbz-name ref/hprc-v1.1-mc-grch38.gbz \
        --ref-paths ref/hprc-v1.1-mc-grch38.paths.sub \
        --in-fq sample_1.fq.gz sample_2.fq.gz \
        --out-bam out/sample.bam


[Parabricks Options Mesg]: Read group for /workdir/sample_1.fq.gz:
[Parabricks Options Mesg]: @RG\tID:sample_rg1\tLB:library\tPL:platform\tSM:sample-name\tPU:pu
[Parabricks Options Mesg]: Marking Duplicates will be run during sorting stages


[PB Info 2026-Mar-02 20:48:11] ------------------------------------------------------------------------------
[PB Info 2026-Mar-02 20:48:11] ||                 Parabricks accelerated Genomics Pipeline                 ||
[PB Info 2026-Mar-02 20:48:11] ||                              Version 4.6.0-1                             ||
[PB Info 2026-Mar-02 20:48:11] ||                              GPU-VG-Giraffe                              ||
[PB Info 2026-Mar-02 20:48:11] ------------------------------------------------------------------------------
[PB Info 2026-Mar-02 20:48:18] Loading indexes
[PB Info 2026-Mar-02 20:49:29] Giraffe CPU loading and initialization: 78.582764 seconds
[PB Info 2026-Mar-02 20:49:29] Using Parabricks batch mapper
[PB Info 2026-Mar-02 20:49:29] Running with 4 GPUs and 3 CUDA streams
[PB Info 2026-Mar-02 20:49:29] Using 64 CPU threads
[PB Info 2026-Mar-02 20:49:29] Writing to intermediate files to be sorted in later stages
[PB Info 2026-Mar-02 20:49:29] Estimating

Please visit https://docs.nvidia.com/clara/#parabricks for detailed documentation




There are new files in the output directory:

- `sample.bam` - the aligned samples 
- `sample.bam.bai` - an index into the aligned samples
- `sample_chrs.txt` - each contig/chromosome and how many reads were mapped to it

In [8]:
%%sh 

ls -lh ../data/out

total 8.3G
-rw-r--r-- 1 root root 3.9G Mar  2 12:09 NIST7035_TAAGGCGA_L001_R1_001.bam
-rw-r--r-- 1 root root 4.0M Mar  2 12:09 NIST7035_TAAGGCGA_L001_R1_001.bam.bai
-rw-r--r-- 1 root root  86K Mar  2 12:09 NIST7035_TAAGGCGA_L001_R1_001_chrs.txt
-rw-r--r-- 1 root root  25M Mar  2 12:10 NIST7035_TAAGGCGA_L001_R1_001.vcf
-rw-r--r-- 1 root root 850K Mar  2 12:09 recal.txt
-rw-r--r-- 1 root root 4.3G Mar  2 12:52 sample.bam
-rw-r--r-- 1 root root 2.8M Mar  2 12:52 sample.bam.bai
-rw-r--r-- 1 root root  536 Mar  2 12:52 sample_chrs.txt


## Pangenome-Aware DeepVariant

In this last section of the notebook, we run variant calling on our aligned samples. 

Since our aligned was run using a pangenome graph, but DeepVariant requires a FASTA reference file, we must create a FASTA from our graph. For this we can again use `vg paths`: 

In [16]:
%%sh 

# Extract the sequences corrresponding to the list of paths to a FASTA file
docker run --rm \
    --volume $(pwd)/../data/ref:/workdir \
    --workdir /workdir \
    quay.io/vgteam/vg:v1.59.0 \
    vg paths \
        -x hprc-v1.1-mc-grch38.gbz \
        -p hprc-v1.1-mc-grch38.paths.sub \
        -F > ../data/ref/hprc-v1.1-mc-grch38.fa

This FASTA needs to be indexed, so we use `samtools faidx` as we would for a normal linear reference. 

In [17]:
%%sh 

samtools faidx ../data/ref/hprc-v1.1-mc-grch38.fa

Now we are ready to run the GPU-accelerated [Parabricks Pangenome-Aware DeepVariant](https://docs.nvidia.com/clara/parabricks/latest/documentation/tooldocs/man_pangenome_aware_deepvariant.html). 

In [18]:
%%sh 

# Run GPU-accelerated Pangenome-Aware DeepVariant using Parabricks
docker run --rm --gpus all \
    --volume $(pwd)/../data:/workdir \
    --workdir /workdir \
    nvcr.io/nvidia/clara/clara-parabricks:4.6.0-1 \
    pbrun pangenome_aware_deepvariant \
    --ref ref/hprc-v1.1-mc-grch38.fa \
    --pangenome ref/hprc-v1.1-mc-grch38.gbz \
    --in-bam out/sample.bam \
    --out-variants out/sample.vcf

[Parabricks Options Mesg]: Setting --num-streams-per-gpu based on available device memory.
[Parabricks Options Mesg]: Please consider using --run-partition for best performance with more than 2 GPUs
Detected 4 CUDA Capable device(s), considering 4 device(s)
  CUDA Driver Version / Runtime Version          13.0 / 12.9
Using model for CUDA Capability Major/Minor version number:    80


[PB Info 2026-Mar-02 21:16:18] ------------------------------------------------------------------------------
[PB Info 2026-Mar-02 21:16:18] ||                 Parabricks accelerated Genomics Pipeline                 ||
[PB Info 2026-Mar-02 21:16:18] ||                              Version 4.6.0-1                             ||
[PB Info 2026-Mar-02 21:16:18] ||                        Pangenome-Aware-Deepvariant                       ||
[PB Info 2026-Mar-02 21:16:18] ------------------------------------------------------------------------------
[PB Info 2026-Mar-02 21:16:18] Starting Pangenome-Aware Deepvariant
[PB Info 2026-Mar-02 21:16:18] Running with 4 GPU devices, each with 2 group instances and 6 workers
[PB Info 2026-Mar-02 21:16:18] Starting constructor, about to open GBZ file: /workdir/ref/hprc-v1.1-mc-grch38.gbz
[PB Info 2026-Mar-02 21:16:18] Loading GBZ file into object. This may take a while for large files...
[PB Info 2026-Mar-02 21:17:02] Loading GBZ file took 189.486843 s

/usr/local/parabricks/binaries/bin/deepsomatic 4 2 --ref /workdir/ref/hprc-v1.1-mc-grch38.fa --reads /workdir/out/sample.bam -o /workdir/out/sample.vcf -n 6 --pangenome /workdir/ref/hprc-v1.1-mc-grch38.gbz --model /usr/local/parabricks/binaries/model/80+/shortread/deepvariant_pangenome_aware.eng -long_reads --channel_insert_size --keep_legacy_allele_counter_behavior --keep_only_window_spanning_haplotypes --keep_supplementary_alignments --min_mapping_quality 0 --normalize_reads --sort_by_haplotypes --parse_sam_aux_fields
Pangenome Aware Deepvariant done, total time: 2.7 min
Please visit https://docs.nvidia.com/clara/#parabricks for detailed documentation



We should now see `sample.vcf` in the output directory. This file contains all the information about which positions in the genome express variations from the reference genome and how those variants were filtered. 

In [19]:
%%sh 

ls -lh ../data/out

total 8.3G
-rw-r--r-- 1 root root 3.9G Mar  2 12:09 NIST7035_TAAGGCGA_L001_R1_001.bam
-rw-r--r-- 1 root root 4.0M Mar  2 12:09 NIST7035_TAAGGCGA_L001_R1_001.bam.bai
-rw-r--r-- 1 root root  86K Mar  2 12:09 NIST7035_TAAGGCGA_L001_R1_001_chrs.txt
-rw-r--r-- 1 root root  25M Mar  2 12:10 NIST7035_TAAGGCGA_L001_R1_001.vcf
-rw-r--r-- 1 root root 850K Mar  2 12:09 recal.txt
-rw-r--r-- 1 root root 4.3G Mar  2 12:52 sample.bam
-rw-r--r-- 1 root root 2.8M Mar  2 12:52 sample.bam.bai
-rw-r--r-- 1 root root  536 Mar  2 12:52 sample_chrs.txt
-rw-r--r-- 1 root root  18M Mar  2 13:18 sample.vcf


We can even inspect the `vcf` using [`bcftools`](https://samtools.github.io/bcftools/bcftools.html) to make sure it was properly written: 

In [20]:
%%sh 

bcftools view --no-header ../data/out/sample.vcf | head

GRCh38#0#chr1	23359	.	C	G	10.1	PASS	.	GT:GQ:DP:AD:VAF:PL	1/1:9:2:0,2:1:9,17,0
GRCh38#0#chr1	23395	.	A	G	6.7	PASS	.	GT:GQ:DP:AD:VAF:PL	1/1:6:2:0,2:1:5,15,0
GRCh38#0#chr1	63268	.	T	C	4.8	PASS	.	GT:GQ:DP:AD:VAF:PL	1/1:5:15:0,15:1:2,17,0
GRCh38#0#chr1	63527	.	T	C	2.6	RefCall	.	GT:GQ:DP:AD:VAF:PL	./.:3:495:369,126:0.254545:0,10,1
GRCh38#0#chr1	63792	.	G	T	19.9	PASS	.	GT:GQ:DP:AD:VAF:PL	1/1:18:258:0,256:0.992248:19,22,0
GRCh38#0#chr1	69453	.	G	A	18.4	PASS	.	GT:GQ:DP:AD:VAF:PL	1/1:16:384:1,382:0.994792:18,20,0
GRCh38#0#chr1	69511	.	A	G	15.3	PASS	.	GT:GQ:DP:AD:VAF:PL	1/1:14:541:1,540:0.998152:15,21,0
GRCh38#0#chr1	69552	.	G	C	15.6	PASS	.	GT:GQ:DP:AD:VAF:PL	1/1:15:528:0,528:1:15,21,0
GRCh38#0#chr1	69569	.	T	C	13.9	PASS	.	GT:GQ:DP:AD:VAF:PL	1/1:13:506:0,506:1:13,20,0
GRCh38#0#chr1	131536	.	C	A	0.5	RefCall	.	GT:GQ:DP:AD:VAF:PL	./.:10:11:9,2:0.181818:0,12,12


## Conclusion 

This notebook demonstrated a complete GPU-accelerated pangenome analysis workflow using NVIDIA Parabricks. We downloaded the HPRC pangenome graph, aligned short-read sequencing samples using the Giraffe aligner, and performed variant calling with Pangenome-Aware DeepVariant. By leveraging pangenome graphs instead of linear reference genomes, this approach captures genetic diversity across populations, leading to improved alignment accuracy and more comprehensive variant detection. The GPU acceleration provided by Parabricks significantly reduces computational time for these computationally intensive genomics tasks.

For more information on GPU accelerated genomics workflows, check out the Parabricks [documentation](https://docs.nvidia.com/clara/parabricks/latest/index.html). 